In [0]:
# Imports and Variable Set Up
import os
import logging
import uuid
import time
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from databricks.vector_search.client import VectorSearchClient

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)
catalog = "workspace"
schema = "ai_project"
volume = "raw_data"
vol_path = f"/Volumes/{catalog}/{schema}/{volume}/"
landing_path = vol_path + "raw"
processed_path = vol_path + "processed"
valid_extensions = ('.txt', '.pdf')

# %pip install -qU databricks-vectorsearch
# dbutils.library.restartPython()

In [0]:
# Check for files & validate size
to_process = []

try:
    raw_files = dbutils.fs.ls(landing_path)
    if not raw_files:
        logger.warning(f"⚠️ Source folder is empty: {landing_path}")
    else:
        logger.info(f"✅ Found {len(raw_files)} files to process.")

        # Validate file size
        for file in raw_files:
            if file.name.lower().endswith(valid_extensions):
                size_kb = file.size / 1024
                if size_kb < 1:
                    logger.error(f"❌ Skipping {file.name}: File is too small ({size_kb:.2f} KB).")
                    continue

                # Store the full path and extension for the next step
                to_process.append({
                    "path": file.path,
                    "name": file.name,
                    "type": "pdf" if file.name.lower().endswith(".pdf") else "text"
                })
                logger.info(f"📖 {file.name} validated ({size_kb:.2f} KB).")
            else:
                logger.warning(f"⚠️  {file} is not a permitted file type.")
        
        if len(to_process) == 0:
            logger.warning(f"⚠️ No valid files found in {landing_path}.")
        else:    
            logger.info(f"✅ Total files ready for ingestion: {len(to_process)}")

except Exception as e:
    logger.error(f"❌ Error accessing volume: {e}")


In [0]:
silver_table = f"{catalog}.{schema}.processed_chunks"

# Set up silver tale and text splitter for chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200,
    add_start_index=True,
    separators=["\n\n", "\n", ". ", " ", ""])

for file_info in to_process:
    try:
        logger.info(f"🚀 Processing: {file_info['name']}")

        local_path = file_info['path'].replace("dbfs:", "")
        
        if file_info['type'] == "pdf":
            loader = PyPDFLoader(local_path)
        else:
            loader = TextLoader(local_path, encoding="utf-8")

        raw_docs = loader.load()

        # Create Chunks
        chunks = text_splitter.split_documents(raw_docs)
        
        # Prepare data for Vector Search with a Unique ID (Required) so the vector index can track specific rows
        data = [{
            "chunk_id": str(uuid.uuid4()),
            "content": chunk.page_content, 
            "source": file_info['name'],
            "type": file_info['type'],
            "page_number": chunk.metadata.get("page", 1), # PDF page number, txt defaults to 1
            "start_index": chunk.metadata.get("start_index", 0) # char position for txt files
        } for chunk in chunks]

        # Convert to Spark DF 
        df = spark.createDataFrame(data)

        # Write to Delta with CDF Enabled
        # Check if table exists, 'overwrite' (first run) else 'append'
        if not spark.catalog.tableExists(silver_table):
            (df.write.format("delta")
               .option("delta.enableChangeDataFeed", "true") # CRITICAL for AI Sync
               .mode("overwrite")
               .saveAsTable(silver_table))
            logger.info(f"✨ Created new table: {silver_table}")
        else:
            df.write.format("delta").mode("append").saveAsTable(silver_table)
            logger.info(f"➕ Appended {len(chunks)} chunks to {silver_table}")

        # Move file to processed folder
        destination = f"{processed_path}/{file_info['name']}"
        dbutils.fs.mv(local_path, destination)
        logger.info(f"✅ Processed and moved: ")

    except Exception as e:
        logger.error(f"❌ Failed to process {file_info['name']}: {e}")

logger.info("🏁 All files processed and synced to Silver Layer.")



In [0]:
# Cell 4: Setup Vector Search Endpoint
endpoint_name = "book_search_endpoint"
vsc = VectorSearchClient()

if vsc.endpoint_exists(endpoint_name):
    logger.info(f"✅ Endpoint '{endpoint_name}' already exists.")
else:
    # Create the endpoint if it doesn't exist
    # This is a 'Standard' endpoint (best for general RAG projects)
    try:
        logger.info(f"🚀 Creating endpoint '{endpoint_name}'.")
        vsc.create_endpoint(name=endpoint_name, endpoint_type="STANDARD")
        vsc.wait_for_endpoint(endpoint_name)
    except Exception as e:
        logger.error(f"❌ Failed to create endpoint: {e}")

# Wait for it to be ready before moving to the next cell
logger.info(f"🟢 Endpoint '{endpoint_name}' is now ONLINE.")

In [0]:
# Unified Cell 5: Idempotent Vector Setup
from databricks.vector_search.client import VectorSearchClient
import time

# vsc = VectorSearchClient()
index_name = f"{catalog}.{schema}.book_vector_index"
embedding_model = "databricks-bge-large-en"

# 1. Ensure Endpoint exists
if not any(e['name'] == endpoint_name for e in vsc.list_endpoints().get('endpoints', [])):
    logger.info(f"🚀 Creating endpoint '{endpoint_name}'...")
    try:
        vsc.create_endpoint(name=endpoint_name, endpoint_type="STANDARD")
        vsc.wait_for_endpoint(endpoint_name)
        logger.info(f"🟢 Endpoint '{endpoint_name}' is now ONLINE.")
    except Exception as e:
        logger.error(f"❌ Failed to create endpoint: {e}")
else:
    logger.info(f"✅ Endpoint '{endpoint_name}' is ready.")

# 2. Ensure Index exists
if not vsc.index_exists(endpoint_name=endpoint_name, index_name=index_name):
    logger.info(f"✨ Creating new index '{index_name}'...")
    try:
        vsc.create_delta_sync_index(
            endpoint_name=endpoint_name,
            source_table_name=silver_table, 
            index_name=index_name,
            pipeline_type="TRIGGERED",
            primary_key="chunk_id",        
            embedding_source_column="content",
            embedding_model_endpoint_name=embedding_model
        )
        # Creation automatically triggers the first sync
        logger.info("⏳ Initial sync started automatically.")
    except Exception as e:
        logger.error(f"❌ Failed to create index: {e}")
else:
    logger.info(f"✅ Index '{index_name}' already exists.")
    
    # 3. Smart Sync: Only sync if we actually processed new files today
    if len(to_process) > 0:
        try:
            logger.info(f"🔄 New data detected ({len(to_process)} files). Triggering sync...")
            vsc.get_index(endpoint_name, index_name).sync()
        except Exception as e:
            logger.error(f"❌ Failed to sync index: {e}")
    else:
        logger.info("ℹ️ No new files in 'raw' folder. Skipping sync to save time.")

# 4. Wait for Index to be queryable (Online)
logger.info("📡 Checking index status...")
while True:
    status = vsc.get_index(endpoint_name, index_name).describe().get("status", {})
    state = status.get("detailed_state", "UNKNOWN")
    if "ONLINE" in state:
        logger.info(f"🟢 Index is ONLINE. Ready for search!")
        break
    elif "FAILED" in state:
        logger.error(f"❌ Index reached a failure state: {state}. Check the Sync History in Catalog Explorer.")
        break
    logger.info(f"⏳ Index state: {state}... (Waiting 30s)")
    time.sleep(30)

In [0]:
# Cell 6: Semantic Search (The Brain Test)
query = "What was the weapon Raskolnikov used in the crime?" 

try:
    # 1. Grab the index object (using the variables you've already defined)
    index = vsc.get_index(endpoint_name, index_name)

    # 2. Search for the top 3 most relevant chunks
    # num_results=3 gives a better 'context' for the AI later
    results = index.similarity_search(
        query_text=query,
        columns=["content", "source", "page_number", "start_index"],
        num_results=4
    )

    docs = results.get('result', {}).get('data_array', [])

    print(f"\n🔍 AI Search Query: '{query}'")
    print("="*70)

    if not docs:
        logger.warning("⚠️ No matches found. The index may still be finalizing its internal mapping.")
    else:
        for i, doc in enumerate(docs):
            # Correcting the indices based on your 'columns' list:
            content = doc[0]
            source = doc[1]
            page = doc[2]
            start_idx = doc[3] # You can print this too if you like!

            print(f"\n📍 [Result {i+1}] | Source: {source} | Page: {page} | Offset: {start_idx}")
            print(f"📄 \"...{content[:450].strip()}...\"")
            print("-" * 30)

except Exception as e:
    logger.error(f"❌ Search failed: {e}")

In [0]:
# Initialize Chat History
if 'chat_history' not in globals():
    chat_history = []
    logger.info("🧠 Memory Initialized.")

In [0]:
from openai import OpenAI
import os

# 1. Setup the Client
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
client = OpenAI(
    api_key=DATABRICKS_TOKEN,
    base_url="https://7474648118426063.ai-gateway.cloud.databricks.com/mlflow/v1"
)

def ask_scholar(question):
    global chat_history
    
    # --- 1. LIBRARIAN: RETRIEVAL ---
    search_results = index.similarity_search(
        query_text=question, 
        columns=["content", "source", "page_number", "start_index"], 
        num_results=4 
    )
    res_data = search_results.get('result', {}).get('data_array', [])
    
    # Build context with your metadata fix
    # row[0] = content
    # row[1] = source
    # row[2] = page_number
    # row[3] = start_index
    context_blocks = [f"Source Page {row[2]} (File: {row[1]}): {row[0]}" for row in res_data]
    context = "\n---\n".join(context_blocks)

    # --- 2. ORCHESTRATION: PREPARE MESSAGES ---
    # Start with the System Prompt (The Rules)
    messages = [
        {
            "role": "system", 
            "content": """You are a literary scholar and forensic analyst specializing in Dostoevsky.
            
            OPERATING RULES:
            1. SCANNING: Search the text for any object used to strike, cut, or kill. Prioritize mentions where an object is associated with blood.
            2. THE 'WHY': Look for mentions of the criminal's mental state, financial condition, or philosophical theories.
            3. EVIDENCE: If you find a weapon, describe the specific actions mentioned in the excerpts.
            
            STRICT CONSTRAINTS:
            4. NO OUTSIDE KNOWLEDGE: Use ONLY the provided excerpts. If the weapon isn't named in the text, say 'The provided excerpts do not name the weapon.'
            5. CITATIONS: Always cite the 'Source Page' and 'File Name' for every piece of information provided.
            6. UNCERTAINTY: If the 'Why' isn't explicitly in the context, explain what IS there (like his poverty) but clarify that a definitive motive isn't stated."""
        }
    ]
    
    # Add the "Past" (The Memory)
    # We use .extend() to add the list of previous turns
    messages.extend(chat_history)
        
    # Add the "Present" (New Context + Current Question)
    user_input = f"NEW CONTEXT EXCERPTS:\n{context}\n\nUSER QUESTION: {question}"
    messages.append({"role": "user", "content": user_input})

    # --- 3. SCHOLAR: GENERATION ---
    response = client.chat.completions.create(
        model="databricks-meta-llama-3-1-405b-instruct",
        messages=messages,
        max_tokens=1024,
        temperature=0.1 # Keep it focused and "scholarly"
    )
    
    answer = response.choices[0].message.content
    
    # --- 4. RECORD: UPDATE HISTORY ---
    # We only store the core Q&A to save tokens
    chat_history.append({"role": "user", "content": question})
    chat_history.append({"role": "assistant", "content": answer})
    
    return answer

print(f"📖 SCHOLAR RESPONSE:\n{ask_scholar('What weapon did Raskolnikov use and why?')}")

In [0]:
print(f"📖 SCHOLAR FOLLOW-UP:\n{ask_scholar('What did he do with the purse and the crosses?')}")

In [0]:
print(ask_scholar("How did his mental state change as he was hiding those items?"))

In [0]:
print(f"📖 SCHOLAR HARD MODE:\n{ask_scholar('Compare Raskolnikov’s intellectual theories to the character Svidrigailov. Does Raskolnikov see a reflection of himself in him, or is he repelled by him?')}")